<a href="https://colab.research.google.com/github/Niroshini2002/pneumonia-xai-gradcam-lime/blob/main/notebooks/ResNet50_Binary_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cell 1: Setup**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install grad-cam lime scikit-image scikit-learn opencv-python-headless -q

# **Cell 2: Dataset**

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q -o chest-xray-pneumonia.zip -d /content/kermany_data
!ls /content/kermany_data/chest_xray

# **Cell 3: Preprocessing + patient-level 80:10:10 split**

In [ ]:
import re, random, numpy as np, cv2, torch
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

IMG_SIZE = 224
DATA_DIR = "/content/kermany_data/chest_xray"
SAVE_DIR = "/content/drive/MyDrive/pneumonia_project/data_binary"
os.makedirs(SAVE_DIR, exist_ok=True)

def patient_id(fn):
    m = re.match(r'(person\d+)', fn)
    if m: return m.group(1)
    m = re.search(r'(IM-\d+)', fn)
    if m: return ('N2-' if fn.startswith('NORMAL2') else '') + m.group(1)
    return fn

X, y, groups = [], [], []
for split in ["train", "val", "test"]:
    for cls, lbl in [("NORMAL", 0), ("PNEUMONIA", 1)]:
        folder = f"{DATA_DIR}/{split}/{cls}"
        for fn in sorted(os.listdir(folder)):
            img = cv2.imread(os.path.join(folder, fn))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            X.append(cv2.resize(img, (IMG_SIZE, IMG_SIZE)))
            y.append(lbl); groups.append(patient_id(fn))

X = np.array(X, dtype=np.uint8); y = np.array(y); groups = np.array(groups)
print(f"Total: {len(X)} | Normal: {(y==0).sum()} | Pneumonia: {(y==1).sum()} | Unique patients/groups: {len(set(groups))}")

# 10% test (group-aware, stratified)
trainval_idx, test_idx = next(StratifiedGroupKFold(10, shuffle=True, random_state=SEED).split(X, y, groups))
# 1/9 = ~10% total val
tr_rel, va_rel = next(StratifiedGroupKFold(9, shuffle=True, random_state=SEED).split(X[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr_rel], trainval_idx[va_rel]

g = lambda idx: set(groups[idx])
assert not (g(train_idx) & g(test_idx)) and not (g(train_idx) & g(val_idx)) and not (g(val_idx) & g(test_idx))
print(f"Train {len(train_idx)} | Val {len(val_idx)} | Test {len(test_idx)}  (patient overlap: 0)")

X_test, y_test = X[test_idx], y[test_idx]
np.savez_compressed(f"{SAVE_DIR}/split.npz", train=train_idx, val=val_idx, test=test_idx)

# **Cell 4: Dataset, model, training functions**

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

normalize = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
augment = T.Compose([T.RandomRotation(10), T.RandomHorizontalFlip(), T.ColorJitter(contrast=0.1)])
BATCH_SIZE = 32

class XrayDataset(Dataset):
    def __init__(self, X, y, train=False):
        self.X, self.y, self.train = X, y, train
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        img = torch.from_numpy(self.X[i]).permute(2, 0, 1).float() / 255.0
        if self.train: img = augment(img)
        return normalize(img), int(self.y[i])

def make_loader(idx, train):
    return DataLoader(XrayDataset(X[idx], y[idx], train), batch_size=BATCH_SIZE, shuffle=train)

def make_model():
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for p in m.parameters(): p.requires_grad = False
    for p in m.layer4.parameters(): p.requires_grad = True      # last block fine-tune
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

def set_train(m):
    m.train()
    for mod in m.modules():     # frozen BatchNorm stats மாறக்கூடாது
        if isinstance(mod, nn.BatchNorm2d) and not any(p.requires_grad for p in mod.parameters()):
            mod.eval()

def evaluate(m, loader):
    m.eval(); trues, preds, probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            out = m(imgs.to(DEVICE))
            probs += torch.softmax(out, 1)[:, 1].cpu().tolist()
            preds += out.argmax(1).cpu().tolist()
            trues += labels.tolist()
    return trues, preds, probs

def run_training(tr_idx, va_idx, epochs, save_path=None):
    m = make_model()
    cw = compute_class_weight("balanced", classes=np.unique(y[tr_idx]), y=y[tr_idx])
    crit = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=1e-4)
    tr_loader = make_loader(tr_idx, True)
    va_loader = make_loader(va_idx, False) if va_idx is not None else None
    hist, best = {"loss": [], "val_acc": [], "val_f1": []}, -1
    for ep in range(epochs):
        set_train(m); total = 0
        for imgs, labels in tr_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad(); loss = crit(m(imgs), labels); loss.backward(); opt.step()
            total += loss.item()
        hist["loss"].append(total / len(tr_loader))
        if va_loader:
            t, p, _ = evaluate(m, va_loader)
            acc, f1 = accuracy_score(t, p), f1_score(t, p)
            hist["val_acc"].append(acc); hist["val_f1"].append(f1)
            print(f"Epoch {ep+1}/{epochs} | Loss {hist['loss'][-1]:.4f} | Val Acc {acc:.4f} | Val F1 {f1:.4f}")
            if f1 > best and save_path:
                best = f1; torch.save(m.state_dict(), save_path); print("  ✔ best saved")
    if save_path and va_loader:
        m.load_state_dict(torch.load(save_path))
    return m, hist

# **Cell 5: Main training**

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/pneumonia_project/resnet50_binary.pth"
model, history = run_training(train_idx, val_idx, epochs=10, save_path=MODEL_PATH)

# **Cell 6: Test evaluation (paper Results table)**

In [ ]:
import matplotlib.pyplot as plt, json
trues, test_preds, probs = evaluate(model, make_loader(test_idx, False))   # shuffle இல்ல → index X_test-ஓட match

results = {
    "accuracy": accuracy_score(trues, test_preds), "precision": precision_score(trues, test_preds),
    "recall": recall_score(trues, test_preds), "f1": f1_score(trues, test_preds),
    "auc_roc": roc_auc_score(trues, probs), "confusion_matrix": confusion_matrix(trues, test_preds).tolist()}
for k, v in results.items(): print(k, ":", v if k == "confusion_matrix" else round(v, 4))

json.dump(results, open("/content/drive/MyDrive/pneumonia_project/test_results.json", "w"), indent=2)

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1); plt.plot(history["loss"]); plt.title("Train loss")
plt.subplot(1, 2, 2); plt.plot(history["val_acc"], label="acc"); plt.plot(history["val_f1"], label="F1"); plt.legend(); plt.title("Validation")
plt.tight_layout(); plt.savefig("/content/drive/MyDrive/pneumonia_project/training_curves.png", dpi=200); plt.show()

# **Cell 7 : 5-fold patient-level CV**

In [ ]:
cv_res = {"acc": [], "f1": [], "auc": []}
for f, (tr, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, groups)):
    print(f"--- Fold {f+1}/5 ---")
    m, _ = run_training(tr, None, epochs=8)
    t, p, pr = evaluate(m, make_loader(te, False))
    cv_res["acc"].append(accuracy_score(t, p)); cv_res["f1"].append(f1_score(t, p)); cv_res["auc"].append(roc_auc_score(t, pr))
    print({k: round(v[-1], 4) for k, v in cv_res.items()})
    del m; torch.cuda.empty_cache()
for k, v in cv_res.items(): print(f"{k}: {np.mean(v):.4f} ± {np.std(v):.4f}")

# **Cell 8: Grad-CAM**

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

model.eval()
cam = GradCAM(model=model, target_layers=[model.layer4[-1]])

def gradcam_for(idx):
    rgb = X_test[idx].astype(np.float32) / 255.0
    tensor = normalize(torch.from_numpy(X_test[idx]).permute(2, 0, 1).float() / 255.0).unsqueeze(0).to(DEVICE)
    pred = model(tensor).argmax(1).item()
    heat = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred)])[0]
    return rgb, heat, show_cam_on_image(rgb, heat, use_rgb=True), pred

# சரியா predict ஆன pneumonia images மட்டும்
good_pneu = [i for i in range(len(y_test)) if y_test[i] == 1 and test_preds[i] == 1]
show_idx = good_pneu[:4]

fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for k, i in enumerate(show_idx):
    rgb, _, overlay, pred = gradcam_for(i)
    ax[0][k].imshow(rgb); ax[0][k].set_title("Original"); ax[0][k].axis("off")
    ax[1][k].imshow(overlay); ax[1][k].set_title(f"Grad-CAM (pred={pred})"); ax[1][k].axis("off")
plt.tight_layout(); plt.savefig("/content/drive/MyDrive/pneumonia_project/gradcam_samples.png", dpi=200); plt.show()

# **Cell 9: LIME (fixed predict_fn)**

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def predict_fn(batch):
    t = torch.from_numpy(np.asarray(batch, dtype=np.float32) / 255.0).permute(0, 3, 1, 2)
    with torch.no_grad():
        return torch.softmax(model(normalize(t).to(DEVICE)), 1).cpu().numpy()

explainer = lime_image.LimeImageExplainer(random_state=SEED)

def lime_heat(i, num_samples=1000):
    exp = explainer.explain_instance(X_test[i], predict_fn, top_labels=2, hide_color=0,
                                     num_samples=num_samples, random_seed=SEED)
    label = exp.top_labels[0]
    heat = np.zeros(exp.segments.shape, dtype=np.float32)
    for s, w in dict(exp.local_exp[label]).items():
        heat[exp.segments == s] = max(w, 0)
    return heat, exp, label

fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for k, i in enumerate(show_idx):
    _, exp, label = lime_heat(i)
    temp, mask = exp.get_image_and_mask(label, positive_only=True, num_features=5, hide_rest=False)
    ax[0][k].imshow(X_test[i]); ax[0][k].set_title("Original"); ax[0][k].axis("off")
    ax[1][k].imshow(mark_boundaries(temp / 255.0, mask)); ax[1][k].set_title(f"LIME (pred={label})"); ax[1][k].axis("off")
plt.tight_layout(); plt.savefig("/content/drive/MyDrive/pneumonia_project/lime_samples.png", dpi=200); plt.show()

# **Cell 10: Quantitative comparison (fixed)**

In [ ]:
import numpy as np, json
from scipy.stats import wilcoxon

# ---------- Helper functions ----------
def top_mask(heat, frac=0.2):
    thr = np.quantile(heat, 1 - frac)
    return (heat >= thr) & (heat > 0)          # zero-heat pixels select aagadhu

def iou(a, b):
    return (a & b).sum() / ((a | b).sum() + 1e-8)

def edge_energy(heat, margin=0.15):
    h, w = heat.shape
    m = int(h * margin)
    inner = np.zeros_like(heat, dtype=bool)
    inner[m:h-m, m:w-m] = True
    return heat[~inner].sum() / (heat.sum() + 1e-8)

rng = np.random.default_rng(SEED)
def random_like(mask):
    r = np.zeros(mask.size, dtype=bool)
    r[rng.permutation(mask.size)[:mask.sum()]] = True
    return r.reshape(mask.shape)

def deletion_drop(idx, mask):
    img = X_test[idx].copy()
    base = predict_fn(img[None])[0, 1]
    img[mask] = 0
    return base - predict_fn(img[None])[0, 1]

# ---------- Random sample (file-order bias illa) ----------
N = 100
rng_sel = np.random.default_rng(SEED)
sel = rng_sel.choice(good_pneu, size=min(N, len(good_pneu)), replace=False).tolist()
print(f"Evaluating {len(sel)} correctly-predicted pneumonia test images")

# ---------- Heatmaps oru thadavai compute pannu (LIME slow, ~13 min) ----------
gc_maps, lime_maps = {}, {}
for n, i in enumerate(sel):
    _, g, _, _ = gradcam_for(i)
    l, _, _ = lime_heat(i)
    gc_maps[i], lime_maps[i] = g, l
    if (n + 1) % 10 == 0:
        print(f"  {n+1}/{len(sel)} done")

# ---------- Primary analysis: top 20% (munnaadiye fix pannina setting) ----------
FRAC = 0.2
res = {k: [] for k in ["iou", "del_gc", "del_lime", "del_rand",
                       "edge_gc", "edge_lime", "area_gc", "area_lime"]}
for i in sel:
    g, l = gc_maps[i], lime_maps[i]
    mg, ml = top_mask(g, FRAC), top_mask(l, FRAC)
    res["iou"].append(iou(mg, ml))
    res["del_gc"].append(deletion_drop(i, mg))
    res["del_lime"].append(deletion_drop(i, ml))
    res["del_rand"].append(deletion_drop(i, random_like(mg)))
    res["edge_gc"].append(edge_energy(g))
    res["edge_lime"].append(edge_energy(l))
    res["area_gc"].append(mg.mean())
    res["area_lime"].append(ml.mean())

print("\n=== Primary results (top 20% regions) ===")
for k, v in res.items():
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    print(f"{k:10s}: mean {np.mean(v):.3f} ± {np.std(v):.3f} | median {med:.3f} (IQR {q1:.3f}-{q3:.3f})")

a, b = np.mean(res["area_gc"]), np.mean(res["area_lime"])
print(f"IoU chance level ~ {a*b/(a+b-a*b):.3f}")

# ---------- Statistical tests (Wilcoxon signed-rank, Bonferroni) ----------
tests = [
    ("Deletion drop: Grad-CAM vs LIME",   res["del_gc"],   res["del_lime"]),
    ("Deletion drop: Grad-CAM vs random", res["del_gc"],   res["del_rand"]),
    ("Deletion drop: LIME vs random",     res["del_lime"], res["del_rand"]),
    ("Edge energy:   Grad-CAM vs LIME",   res["edge_gc"],  res["edge_lime"]),
]
alpha = 0.05 / len(tests)
print(f"\n=== Wilcoxon signed-rank (Bonferroni alpha = {alpha:.4f}) ===")
pvals = {}
for name, x, y_ in tests:
    try:
        p = wilcoxon(x, y_).pvalue
    except ValueError:
        p = float("nan")
    pvals[name] = p
    print(f"{name}: p = {p:.4f} {'(significant)' if p < alpha else '(not significant)'}")

# ---------- Sensitivity check: 10/20/30% (ellaa results-um report pannunga) ----------
print("\n=== Sensitivity: deletion drop vs mask size ===")
sens = {}
for f in [0.10, 0.20, 0.30]:
    dg, dl, dr = [], [], []
    for i in sel:
        mg, ml = top_mask(gc_maps[i], f), top_mask(lime_maps[i], f)
        dg.append(deletion_drop(i, mg))
        dl.append(deletion_drop(i, ml))
        dr.append(deletion_drop(i, random_like(mg)))
    sens[f] = dict(gradcam=float(np.mean(dg)), lime=float(np.mean(dl)), random=float(np.mean(dr)))
    print(f"top {int(f*100):2d}%: Grad-CAM {np.mean(dg):.3f} | LIME {np.mean(dl):.3f} | random {np.mean(dr):.3f}")

# ---------- Drive-la save ----------
out = {
    "n_images": len(sel),
    "means": {k: float(np.mean(v)) for k, v in res.items()},
    "medians": {k: float(np.median(v)) for k, v in res.items()},
    "p_values": pvals,
    "bonferroni_alpha": alpha,
    "sensitivity": {str(k): v for k, v in sens.items()},
}
json.dump(out, open("/content/drive/MyDrive/pneumonia_project/xai_comparison.json", "w"), indent=2)
print("\nSaved xai_comparison.json")